In [30]:
from bs4 import BeautifulSoup
import requests
import pandas as pd

URL = "https://finance.yahoo.com/markets/crypto/most-active/"
HEADERS = {"User-Agent": "Mozilla/5.0"}

def parse_human_readable(value):
    if not value:
        return None
    val = value.replace(',', '')
    last_char = val[-1]
    val_no_last_char = val[:-1]
    match(last_char):
        case 'K':
            return float(val_no_last_char) * 1000
        case 'M':
            return float(val_no_last_char) * 1_000_000
        case 'B':
            return float(val_no_last_char) * 1_000_000_000
        case 'T':
            return float(val_no_last_char) * 1_000_000_000_000
        case _:
            return float(val)


def crypto_volumes():
    response = requests.get(URL, headers=HEADERS)
    print(response)
    html = response.text
    soup = BeautifulSoup(html, "html.parser")
    table = soup.find("table", attrs={"class": "yf-19xszjc"})
    #print(table)
    rows = table.find('tbody').find_all('tr')
    #print(rows)
    data = []
    for row in rows:
        #print(row)
        print('---------------------')
        cells = row.find_all('td')

        symbol = cells[0].text.strip()
        name = cells[1].text.strip()
        price = cells[4].text.strip()
        market_cap = cells[6].text.strip()
        volume = cells[8].text.strip()
        record = {
            'symbol': symbol,
            'name': name,
            'price': parse_human_readable(price),
            'market_cap': parse_human_readable(market_cap),
            'volume': parse_human_readable(volume)
        }
        print(record)
        data.append(record)
    df = pd.DataFrame(data)
    df = df.set_index('symbol')
    return df

def save_crypto_volumes(df):
    print(df)
    df.to_csv('data/crypto_volumes.csv', index=True)
    df.to_excel('data/crypto_volumes.xlsx', index=True)
    df.to_json('data/crypto_volumes.json', orient='records', indent=4)
    #df.to_json('data/crypto_volumes.json', index=False)



In [32]:
df = crypto_volumes()
save_crypto_volumes(df)

<Response [200]>
---------------------
{'symbol': 'USDT-USD', 'name': 'Tether USDt USD', 'price': 0.0, 'market_cap': 176854000000.0, 'volume': 55430000000.0}
---------------------
{'symbol': 'BTC-USD', 'name': 'Bitcoin USD', 'price': -1641.42, 'market_cap': 1492000000000.0, 'volume': 25266000000.0}
---------------------
{'symbol': 'ETH-USD', 'name': 'Ethereum USD', 'price': -125.99, 'market_cap': 257394999999.99997, 'volume': 14083000000.0}
---------------------
{'symbol': 'USDC-USD', 'name': 'USDC USD', 'price': -0.0, 'market_cap': 70235000000.0, 'volume': 9790000000.0}
---------------------
{'symbol': 'SOL-USD', 'name': 'Solana USD', 'price': -2.65, 'market_cap': 51071000000.0, 'volume': 2702000000.0}
---------------------
{'symbol': 'XRP-USD', 'name': 'XRP USD', 'price': -0.07, 'market_cap': 69199000000.0, 'volume': 2628000000.0}
---------------------
{'symbol': 'BNB-USD', 'name': 'BNB USD', 'price': -16.05, 'market_cap': 89505000000.0, 'volume': 1523000000.0}
---------------------
